In [19]:
!pip install streamlit pandas numpy matplotlib scikit-learn lightgbm shap joblib imbalanced-learn utils

In [20]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st


def inject_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&display=swap');
    html, body, [class*="css"] { font-family: 'DM Sans', sans-serif; }

    [data-testid="stSidebar"] { background: #0f1117; border-right: 1px solid #1e2130; }
    [data-testid="stSidebar"] * { color: #e0e6f0 !important; }

    div[data-testid="metric-container"] {
        background: #f8faff; border: 1px solid #dde3f0;
        border-radius: 10px; padding: 14px 18px;
        box-shadow: 0 2px 8px rgba(0,60,180,0.06);
    }
    .risk-high {
        background: #fff0f0; border: 2px solid #e53e3e; border-radius: 12px;
        padding: 20px 28px; text-align: center; color: #c53030;
        font-weight: 700; font-size: 1.5rem;
    }
    .risk-low {
        background: #f0fff4; border: 2px solid #38a169; border-radius: 12px;
        padding: 20px 28px; text-align: center; color: #276749;
        font-weight: 700; font-size: 1.5rem;
    }
    .info-box {
        background: #eef3ff; border-left: 4px solid #3b6fdb;
        border-radius: 0 8px 8px 0; padding: 10px 16px;
        font-size: 0.9rem; color: #1a2a5e; margin: 8px 0 16px 0;
    }
    </style>
    """, unsafe_allow_html=True)


@st.cache_resource
def load_models():
    needed = {
        "lgbm":      "lgbm_model.pkl",
        "top_feats": "top_features.pkl",
        "threshold": "threshold.json",
    }
    artefacts, missing = {}, []
    for key, fname in needed.items():
        if os.path.exists(fname):
            if fname.endswith(".json"):
                with open(fname) as f:
                    artefacts[key] = json.load(f)
            else:
                artefacts[key] = joblib.load(fname)
        else:
            missing.append(fname)
    return artefacts, missing


def build_input(raw: dict, top_feats: list) -> pd.DataFrame:
    row = {f: 0.0 for f in top_feats}
    for k, v in raw.items():
        if k in row:
            row[k] = v
    return pd.DataFrame([row])[top_feats]


def gauge_chart(prob: float):
    fig, ax = plt.subplots(figsize=(4, 2.2))
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.2, 1.1); ax.axis("off")
    theta_bg = np.linspace(np.pi, 0, 200)
    ax.plot(np.cos(theta_bg), np.sin(theta_bg), color="#e0e6f0", lw=18, solid_capstyle="round")
    end_a = np.pi - prob * np.pi
    theta_fg = np.linspace(np.pi, end_a, 200)
    col = "#e53e3e" if prob > 0.5 else "#d97706" if prob > 0.3 else "#38a169"
    ax.plot(np.cos(theta_fg), np.sin(theta_fg), color=col, lw=18, solid_capstyle="round")
    ax.annotate("", xy=(np.cos(end_a) * 0.75, np.sin(end_a) * 0.75), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="#1a2240", lw=2.5))
    ax.plot(0, 0, "o", color="#1a2240", ms=7, zorder=5)
    ax.text(0, -0.15, f"{prob:.1%}", ha="center", fontsize=18, fontweight="bold", color=col)
    ax.text(0,  0.55, "Default\nProbability", ha="center", fontsize=8, color="#8b9bbf")
    fig.patch.set_alpha(0)
    return fig

In [21]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings("ignore")


# Removed for Colab notebook execution: from utils import build_input, gauge_chart


def predict_show(artefacts: dict):
    st.markdown("# 🔍 Loan Default Risk Prediction")
    st.markdown(
        '<div class="info-box">Fill in the applicant\'s financial profile. '
        'The LightGBM model (best recall) will predict default risk.</div>',
        unsafe_allow_html=True
    )

    top_feats = artefacts["top_feats"]
    threshold = artefacts["threshold"]["threshold"]

    with st.form("applicant_form"):
        c1, c2, c3 = st.columns(3)

        with c1:
            st.markdown("**💰 Income & Credit**")
            AMT_INCOME_TOTAL = st.number_input("Annual Income",   10_000, 10_000_000, 200_000, step=5_000)
            AMT_CREDIT       = st.number_input("Loan Amount",     10_000, 10_000_000, 500_000, step=10_000)
            AMT_ANNUITY      = st.number_input("Monthly Annuity",  1_000,    500_000,  25_000, step=1_000)
            AMT_GOODS_PRICE  = st.number_input("Goods Price",          0, 10_000_000, 450_000, step=10_000)

        with c2:
            st.markdown("**👤 Personal**")
            AGE_YEARS       = st.slider("Age (years)", 18, 70, 35)
            YEARS_EMPLOYED  = st.slider("Years Employed", 0, 40, 5)
            CNT_FAM_MEMBERS = st.slider("Family Members", 1, 10, 3)
            EXT_SOURCE_2    = st.slider("External Score 2 (0–1)", 0.0, 1.0, 0.55, 0.01)
            EXT_SOURCE_3    = st.slider("External Score 3 (0–1)", 0.0, 1.0, 0.50, 0.01)

        with c3:
            st.markdown("**📋 Loan Details**")
            REGION_RATING   = st.selectbox("Region Risk Rating", [1, 2, 3])
            FLAG_OWN_CAR    = st.selectbox("Owns a Car?", ["Y", "N"])
            FLAG_OWN_REALTY = st.selectbox("Owns Realty?", ["Y", "N"])

        submitted = st.form_submit_button("⚡ Predict Default Risk", use_container_width=True)

    if not submitted:
        return

    # Engineered features
    CREDIT_INCOME_RATIO   = AMT_CREDIT      / (AMT_INCOME_TOTAL + 1)
    ANNUITY_INCOME_RATIO  = AMT_ANNUITY     / (AMT_INCOME_TOTAL + 1)
    GOODS_CREDIT_RATIO    = AMT_GOODS_PRICE / (AMT_CREDIT + 1)
    CREDIT_TERM           = AMT_CREDIT      / (AMT_ANNUITY + 1)
    EMPLOYED_TO_AGE_RATIO = YEARS_EMPLOYED  / (AGE_YEARS + 1)
    INCOME_PER_PERSON     = AMT_INCOME_TOTAL / (CNT_FAM_MEMBERS + 1)

    raw = {
        "AMT_INCOME_TOTAL":            AMT_INCOME_TOTAL,
        "AMT_CREDIT":                  AMT_CREDIT,
        "AMT_ANNUITY":                 AMT_ANNUITY,
        "AMT_GOODS_PRICE":             AMT_GOODS_PRICE,
        "DAYS_BIRTH":                  -int(AGE_YEARS * 365),
        "DAYS_EMPLOYED":               -int(YEARS_EMPLOYED * 365),
        "AGE_YEARS":                   AGE_YEARS,
        "YEARS_EMPLOYED":              YEARS_EMPLOYED,
        "CNT_FAM_MEMBERS":             CNT_FAM_MEMBERS,
        "EXT_SOURCE_2":                EXT_SOURCE_2,
        "EXT_SOURCE_3":                EXT_SOURCE_3,
        "CREDIT_INCOME_RATIO":         CREDIT_INCOME_RATIO,
        "ANNUITY_INCOME_RATIO":        ANNUITY_INCOME_RATIO,
        "GOODS_CREDIT_RATIO":          GOODS_CREDIT_RATIO,
        "CREDIT_TERM":                 CREDIT_TERM,
        "EMPLOYED_TO_AGE_RATIO":       EMPLOYED_TO_AGE_RATIO,
        "INCOME_PER_PERSON":           INCOME_PER_PERSON,
        "REGION_RATING_CLIENT":        REGION_RATING,
        "REGION_RATING_CLIENT_W_CITY": REGION_RATING,
    }

    X_in  = build_input(raw, top_feats)
    prob  = artefacts["lgbm"].predict_proba(X_in)[0][1]
    pred  = int(prob >= threshold)

    st.markdown("---")

    # Verdict + gauge
    st.markdown("### 🎯 Prediction Result")
    col_v, col_g = st.columns([1.3, 1])
    with col_v:
        if pred == 1:
            st.markdown(
                f'<div class="risk-high">⚠️ HIGH RISK — LIKELY DEFAULT<br>'
                f'<span style="font-size:0.9rem;font-weight:400">'
                f'Probability {prob:.1%} ≥ threshold {threshold:.3f}</span></div>',
                unsafe_allow_html=True)
        else:
            st.markdown(
                f'<div class="risk-low">✅ LOW RISK — LIKELY SAFE<br>'
                f'<span style="font-size:0.9rem;font-weight:400">'
                f'Probability {prob:.1%} < threshold {threshold:.3f}</span></div>',
                unsafe_allow_html=True)
    with col_g:
        st.pyplot(gauge_chart(prob), use_container_width=True)

    # Metric
    st.metric("Default Probability", f"{prob:.1%}", delta="⚠️ DEFAULT" if pred else "✅ SAFE")

    # Engineered features breakdown
    with st.expander("📐 Engineered Features Used"):
        st.table(pd.DataFrame({
            "Feature": [
                "Credit-Income Ratio", "Annuity-Income Ratio", "Goods-Credit Ratio",
                "Credit Term (months≈)", "Employed-Age Ratio", "Income per Person",
            ],
            "Value": [
                f"{CREDIT_INCOME_RATIO:.3f}", f"{ANNUITY_INCOME_RATIO:.3f}",
                f"{GOODS_CREDIT_RATIO:.3f}",  f"{CREDIT_TERM:.1f}",
                f"{EMPLOYED_TO_AGE_RATIO:.3f}", f"{INCOME_PER_PERSON:,.0f}",
            ],
        }).set_index("Feature"))

    # Local SHAP waterfall
    st.markdown("### 🧠 SHAP Explanation")
    with st.spinner("Computing SHAP..."):
        try:
            exp_shap = shap.TreeExplainer(artefacts["lgbm"])
            sv_in    = exp_shap.shap_values(X_in)
            sv_in    = sv_in[1] if isinstance(sv_in, list) else sv_in
            base_val = (exp_shap.expected_value[1]
                        if isinstance(exp_shap.expected_value, list)
                        else exp_shap.expected_value)
            explanation = shap.Explanation(
                values        = sv_in[0],
                base_values   = base_val,
                data          = X_in.iloc[0].values,
                feature_names = top_feats,
            )
            shap.waterfall_plot(explanation, max_display=12, show=False)
            plt.title("SHAP Waterfall — This Applicant", fontweight="bold")
            plt.tight_layout()
            st.pyplot(plt.gcf(), use_container_width=True)
            plt.close("all")
        except Exception as e:
            st.info(f"SHAP unavailable: {e}")

In [22]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score
)


def evaluation_show(artefacts: dict):
    st.markdown("# 📊 Model Evaluation")
    st.markdown(
        '<div class="info-box">Upload your evaluation CSV '
        '(columns: <code>y_true, y_proba_lgbm</code>).</div>',
        unsafe_allow_html=True
    )
    st.code(
        "pd.DataFrame({\n"+
        "    'y_true':       y_test.values,\n"+
        "    'y_proba_lgbm': y_proba_lgbm,\n"+
        "}).to_csv('eval_results.csv', index=False)",
        language="python"
    )

    uploaded = st.file_uploader("Upload eval_results.csv", type=["csv"])
    if not uploaded:
        return

    df_e = pd.read_csv(uploaded)
    if not {"y_true", "y_proba_lgbm"}.issubset(df_e.columns):
        st.error("CSV must contain columns: y_true, y_proba_lgbm")
        return

    thr  = artefacts["threshold"]["threshold"]
    yt   = df_e["y_true"]
    prob = df_e["y_proba_lgbm"]
    pred = (prob >= thr).astype(int)

    st.markdown("---")

    # Metrics
    st.markdown("### 📋 Summary Metrics")
    ca, cb, cc, cd, ce = st.columns(5)
    ca.metric("Accuracy",  f"{accuracy_score(yt, pred):.4f}")
    cb.metric("Precision", f"{precision_score(yt, pred, zero_division=0):.4f}")
    cc.metric("Recall",    f"{recall_score(yt, pred):.4f}")
    cd.metric("F1-Score",  f"{f1_score(yt, pred):.4f}")
    ce.metric("ROC-AUC",   f"{roc_auc_score(yt, prob):.4f}")

    # Confusion matrix
    st.markdown("### 🔲 Confusion Matrix")
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(yt, pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Default", "Default"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("LightGBM — Confusion Matrix", fontweight="bold")
    plt.tight_layout()
    st.pyplot(fig, use_container_width=False)
    plt.close("all")

    # ROC curve
    st.markdown("### 📈 ROC-AUC Curve")
    fig, ax = plt.subplots(figsize=(7, 5))
    fpr, tpr, _ = roc_curve(yt, prob)
    auc = roc_auc_score(yt, prob)
    ax.plot(fpr, tpr, color="#1a56db", lw=2, label=f"LightGBM (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC-AUC Curve", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout()
    st.pyplot(fig, use_container_width=True)
    plt.close("all")

    # Precision-Recall curve
    st.markdown("### 📉 Precision-Recall Curve")
    fig, ax = plt.subplots(figsize=(7, 5))
    prec, rec, _ = precision_recall_curve(yt, prob)
    ax.plot(rec, prec, color="#1a56db", lw=2, label="LightGBM")
    ax.axhline(yt.mean(), color="red", ls="--", label=f"Baseline ({yt.mean():.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout()
    st.pyplot(fig, use_container_width=True)
    plt.close("all")

    st.info(f"Decision threshold: **{thr:.4f}** (tuned for Recall ≥ 0.70)")

In [23]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings("ignore")


def explainability_show(artefacts: dict):
    st.markdown("# 🧠 Model Explainability (SHAP)")
    st.markdown(
        '<div class="info-box">Upload <code>X_test.csv</code> (top-50 feature columns) '
        'to render global SHAP plots.</div>',
        unsafe_allow_html=True
    )
    st.code("X_test.to_csv('X_test.csv', index=False)", language="python")

    top_feats  = artefacts["top_feats"]
    uploaded_X = st.file_uploader("Upload X_test.csv", type=["csv"])

    if not uploaded_X:
        _show_shap_info()
        return

    X_t = pd.read_csv(uploaded_X)
    for c in top_feats:
        if c not in X_t.columns:
            X_t[c] = 0.0
    X_t = X_t[top_feats]

    n = st.slider("SHAP sample size", 100, min(2000, len(X_t)), 500, 100)

    if not st.button("⚡ Compute SHAP Values", use_container_width=True):
        return

    with st.spinner("Computing SHAP values..."):
        try:
            Xs   = X_t.sample(n=n, random_state=42)
            expl = shap.TreeExplainer(artefacts["lgbm"])
            svs  = expl.shap_values(Xs)
            sv   = svs[1] if isinstance(svs, list) else svs
            mi   = np.abs(sv).mean(axis=0)
        except Exception as e:
            st.error(f"SHAP failed: {e}")
            return

    st.markdown("---")

    # Global bar
    st.markdown("### 📊 Global Feature Importance (Mean |SHAP|)")
    fig, ax = plt.subplots(figsize=(9, 6))
    pd.Series(mi, index=top_feats).sort_values().tail(20).plot(
        kind="barh", ax=ax, color="#1a56db", edgecolor="white")
    ax.set_title("LightGBM — Top 20 by Mean |SHAP|", fontweight="bold")
    ax.set_xlabel("Mean |SHAP Value|")
    plt.tight_layout()
    st.pyplot(fig, use_container_width=True)
    plt.close("all")

    # Beeswarm
    st.markdown("### 🐝 Beeswarm Plot (Direction & Magnitude)")
    fig2 = plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, Xs, max_display=20, show=False)
    plt.title("LightGBM — SHAP Beeswarm", fontweight="bold")
    plt.tight_layout()
    st.pyplot(fig2, use_container_width=True)
    plt.close("all")

    # Dependence plots
    st.markdown("### 🔗 Dependence Plots (Top 3 Features)")
    top3 = Xs.columns[np.argsort(mi)[::-1][:3]].tolist()
    fig3, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, ft in zip(axes, top3):
        shap.dependence_plot(ft, sv, Xs, ax=ax, show=False)
        ax.set_title(ft, fontsize=10)
    plt.suptitle("SHAP Dependence Plots — Top 3 Features", fontweight="bold", y=1.02)
    plt.tight_layout()
    st.pyplot(fig3, use_container_width=True)
    plt.close("all")

    # Individual waterfalls
    st.markdown("### 👤 Individual Applicant Explanations")
    base_val    = (expl.expected_value[1] if isinstance(expl.expected_value, list)
                   else expl.expected_value)
    y_preds     = artefacts["lgbm"].predict(Xs)
    default_idx = np.where(y_preds == 1)[0]
    safe_idx    = np.where(y_preds == 0)[0]

    col_d, col_s = st.columns(2)
    with col_d:
        st.markdown("**Predicted Defaulter**")
        if len(default_idx) > 0:
            idx = default_idx[0]
            exp = shap.Explanation(values=sv[idx], base_values=base_val,
                                   data=Xs.iloc[idx].values, feature_names=top_feats)
            shap.waterfall_plot(exp, max_display=10, show=False)
            plt.title("Waterfall — Defaulter", fontweight="bold")
            plt.tight_layout()
            st.pyplot(plt.gcf(), use_container_width=True)
            plt.close("all")
        else:
            st.info("No defaulters in this sample.")

    with col_s:
        st.markdown("**Predicted Non-Defaulter**")
        if len(safe_idx) > 0:
            idx = safe_idx[0]
            exp = shap.Explanation(values=sv[idx], base_values=base_val,
                                   data=Xs.iloc[idx].values, feature_names=top_feats)
            shap.waterfall_plot(exp, max_display=10, show=False)
            plt.title("Waterfall — Non-Defaulter", fontweight="bold")
            plt.tight_layout()
            st.pyplot(plt.gcf(), use_container_width=True)
            plt.close("all")
        else:
            st.info("No non-defaulters in this sample.")

    st.success("✅ SHAP analysis complete!")


def _show_shap_info():
    col_a, col_b = st.columns(2)
    with col_a:
        st.markdown("""
**What is SHAP?**
Assigns each feature a contribution value per prediction using game theory.
- Locally accurate & model-agnostic
- Satisfies EU AI Act transparency requirements
- TreeExplainer gives exact results for LightGBM
        """)
    with col_b:
        st.markdown("""
| Plot | What it shows |
|------|--------------|
| Bar | Mean feature importance |
| Beeswarm | Direction & magnitude per sample |
| Dependence | How one feature drives SHAP |
| Waterfall | Single-applicant breakdown |

High `EXT_SOURCE_2/3` → ↓ risk · High `CREDIT_INCOME_RATIO` → ↑ risk
        """)

In [24]:
import streamlit as st


def about_show():
    st.markdown("# ℹ️ Project #11 — Loan Default Risk Classification")

    st.markdown("### 🔁 Full Pipeline")
    st.markdown("""
| Stage | Description |
|-------|-------------|
| 1–4   | Data loading, EDA, cleaning, preprocessing |
| 5     | Feature engineering (8 ratios) + top 50 by mutual information |
| 6     | Model training: Logistic Regression, Random Forest, LightGBM |
| 7     | Confusion matrices, ROC-AUC, PR curves, 5-fold CV, threshold tuning |
| 8     | SHAP: global summary, beeswarm, dependence, force/waterfall plots |
| **9** | **This Streamlit deployment** |
    """)

    st.markdown("---")
    st.markdown("### 📁 Required Files")
    st.code("lgbm_model.pkl\ntop_features.pkl\nthreshold.json", language="text")

    st.markdown("### 📤 Export from Notebook")
    st.code(
        "import joblib, json\n"+
        "joblib.dump(lgbm,      'lgbm_model.pkl')\n"+
        "joblib.dump(top_feats, 'top_features.pkl')\n"+
        "with open('threshold.json','w') as f:\n"+
        "    json.dump({'threshold': float(THRESHOLD)}, f)",
        language="python"
    )

    st.markdown("---")
    st.markdown("### 🚀 Run")
    st.code("pip install streamlit pandas numpy matplotlib scikit-learn lightgbm shap joblib\nstreamlit run app.py", language="bash")

    st.markdown("---")
    st.markdown("### 🎯 Key Design Choices")
    st.markdown("""
- **Recall ≥ 0.70** — missed defaulters cost more than false alarms
- **SMOTE on training only** — no data leakage
- **LightGBM** — best ROC-AUC and recall at tuned threshold
- **SHAP TreeExplainer** — exact, fast, regulatory-compliant
    """)

In [25]:
import streamlit as st
# from utils import inject_css, load_models # Removed for Colab notebook execution
# from pages import predict, evaluation, explainability, about # Removed for Colab notebook execution

st.set_page_config(
    page_title="Loan Default Risk Classifier",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_css()

artefacts, missing_files = load_models()
models_loaded = len(missing_files) == 0

with st.sidebar:
    st.markdown("## 🏦 Loan Default\nRisk Classifier")
    st.markdown("---")
    page = st.radio("Navigate", [
        "🔍 Predict Risk",
        "📊 Model Evaluation",
        "🧠 Explainability",
        "ℹ️ About",
    ], label_visibility="collapsed")

    st.markdown("---")
    st.markdown("**Model Status**")
    if models_loaded:
        threshold = artefacts["threshold"]["threshold"]
        st.markdown("<small style='color:#6ee7b7'>● LightGBM ✓</small>", unsafe_allow_html=True)
        st.markdown(f"<small style='color:#8b9bbf'>Threshold: <strong>{threshold:.4f}</strong></small>",
                    unsafe_allow_html=True)
    else:
        st.error("Model files missing")
        for f in missing_files:
            st.markdown(f"<small style='color:#fc8181'>✗ {f}</small>", unsafe_allow_html=True)

if page == "🔍 Predict Risk":
    if models_loaded:
        predict_show(artefacts)
    else:
        st.error("Add lgbm_model.pkl, top_features.pkl, threshold.json to the app folder.")

elif page == "📊 Model Evaluation":
    if models_loaded:
        evaluation_show(artefacts)
    else:
        st.error("Models not loaded.")

elif page == "🧠 Explainability":
    if models_loaded:
        explainability_show(artefacts)
    else:
        st.error("Models not loaded.")

elif page == "ℹ️ About":
    about_show()

2026-05-15 11:53:58.611 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.613 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.614 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 11:53:58.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar